<a href="https://colab.research.google.com/github/tien10022001/txldl/blob/main/lab4_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import re
import os
import nltk
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

# Tải tài nguyên tách từ
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

class Lab4Processor:
    def __init__(self, file_path, search_word):
        self.file_path = file_path
        self.search_word = search_word
        self.stop_words = ["là", "và", "có", "rất", "thì", "một", "những", "các", "ở"]
        self.df = None
        self.text_col = None

    def clean_text(self, text):
        """Tiền xử lý văn bản: lowercase, xóa dấu câu, tokenize, lọc stop words"""
        text = str(text).lower()
        text = re.sub(r"[^\w\s]", " ", text)
        tokens = word_tokenize(text)
        return [w for w in tokens if w not in self.stop_words]

    def _detect_text_column(self):
        """Tự động xác định cột chứa review/comment/feedback"""
        keywords = ['review', 'comment', 'feedback', 'text', 'nội dung']
        for col in self.df.columns:
            if any(k in col.lower() for k in keywords):
                return col
        return self.df.select_dtypes(include=['object']).columns[0]

    def run(self):
        if not os.path.exists(self.file_path):
            print(f"❌ Không tìm thấy file: {self.file_path}")
            return

        print(f"\n{'='*20} Đang xử lý: {self.file_path} {'='*20}")

        # 1. Nạp dữ liệu & Xử lý missing values
        self.df = pd.read_csv(self.file_path, encoding='utf-8-sig')
        self.text_col = self._detect_text_column()
        self.df.dropna(subset=[self.text_col], inplace=True)
        self.df.fillna("Unknown", inplace=True)

        # 2. Label Encoding các cột categorical
        le = LabelEncoder()
        cat_cols = self.df.select_dtypes(include=['object']).columns.tolist()
        if self.text_col in cat_cols: cat_cols.remove(self.text_col)
        for col in cat_cols:
            self.df[col] = le.fit_transform(self.df[col].astype(str))

        # 3. Tiền xử lý văn bản
        self.df['tokens'] = self.df[self.text_col].apply(self.clean_text)
        clean_docs = self.df['tokens'].apply(lambda x: " ".join(x))

        # 4. TF-IDF Matrix
        tfidf = TfidfVectorizer()
        tfidf_matrix = tfidf.fit_transform(clean_docs)
        print(f"✅ Ma trận TF-IDF: {tfidf_matrix.shape}")

        # 5. Word2Vec Embeddings
        w2v_model = Word2Vec(sentences=self.df['tokens'].tolist(), vector_size=100, window=5, min_count=1)

        # Tìm từ gần nghĩa
        print(f"🔍 5 từ gần nghĩa nhất với '{self.search_word}':")
        try:
            # Lấy token đầu tiên để tìm kiếm chính xác trong vocab
            term = self.search_word.split()[0]
            sims = w2v_model.wv.most_similar(term, topn=5)
            for word, score in sims:
                print(f"   - {word}: {score:.4f}")
        except KeyError:
            print(f"   ⚠️ Từ '{self.search_word}' không xuất hiện trong dữ liệu.")

# --- THỰC THI 4 BÀI TẬP ---
tasks = [
    ("ITA105_Lab_4_Hotel_reviews.csv", "sạch"),
    ("ITA105_Lab_4_Match_comments.csv", "xuất"),
    ("ITA105_Lab_4_Player_feedback.csv", "đẹp"),
    ("ITA105_Lab_4_Album_reviews.csv", "sáng")
]

for file, word in tasks:
    processor = Lab4Processor(file, word)
    processor.run()


==================== Đang xử lý: ITA105_Lab_4_Hotel_reviews.csv ====================
✅ Ma trận TF-IDF: (147, 38)
🔍 5 từ gần nghĩa nhất với 'sạch':
   - trí: 0.2268
   - khá: 0.2163
   - tắm: 0.1948
   - mềm: 0.1938
   - chưa: 0.1833

==================== Đang xử lý: ITA105_Lab_4_Match_comments.csv ====================
✅ Ma trận TF-IDF: (196, 47)
🔍 5 từ gần nghĩa nhất với 'xuất':
   - thi: 0.2821
   - mắt: 0.2547
   - sắc: 0.2535
   - thủ: 0.2508
   - nhưng: 0.2200

==================== Đang xử lý: ITA105_Lab_4_Player_feedback.csv ====================
✅ Ma trận TF-IDF: (175, 32)
🔍 5 từ gần nghĩa nhất với 'đẹp':
   - dẫn: 0.2295
   - thú: 0.2112
   - ứng: 0.1857
   - gameplay: 0.1807
   - hấp: 0.1646

==================== Đang xử lý: ITA105_Lab_4_Album_reviews.csv ====================
✅ Ma trận TF-IDF: (156, 39)
🔍 5 từ gần nghĩa nhất với 'sáng':
   - phong: 0.2347
   - hát: 0.2164
   - thiếu: 0.2099
   - lắng: 0.2018
   - xúc: 0.1764
